# Data Cleaning & Validation

## Purpose
Profile both raw files, fix data-quality problems, check that the two tables join correctly, and save analysis-ready versions for the rest of the project.

## Principles
- **Raw data is never modified.** All fixes happen on copies.
- **Every change is logged** with the number of rows affected (`reports/cleaning_log.md`).
- **Ambiguous problems are flagged, not silently "fixed"**, so the decision can be tested later in a sensitivity analysis.

In [22]:
import pandas as pd
import numpy as np
from pathlib import Path

pd.set_option("display.width", 160)
pd.set_option("display.max_columns", 40)

RAW = Path("../data/raw")
PROC = Path("../data/processed")
REPORTS = Path("../reports")
PROC.mkdir(exist_ok=True)
REPORTS.mkdir(exist_ok=True)

DATA_END = pd.Timestamp("2025-12-31")   # last day covered by the data

log = []   # each entry: (area, issue, rows affected, action)

def record(area, issue, n, action):
    log.append((area, issue, int(n), action))
    print(f"[{area}] {issue}: {int(n):,} rows -> {action}")

Data loading

In [23]:
subs_raw = pd.read_csv(RAW / "subscriptions.csv", dtype=str, keep_default_na=False)
usage_raw = pd.read_csv(RAW / "usage.csv", dtype=str, keep_default_na=False)

print("subscriptions:", subs_raw.shape, "| usage:", usage_raw.shape)
assert subs_raw.shape == (4200, 11) and usage_raw.shape == (30685, 6)

subs_raw.head()

subscriptions: (4200, 11) | usage: (30685, 6)


,customer_id,signup_date,plan_type,plan_tier,monthly_price,acquisition_channel,primary_device,age_group,churned,churn_date,cancel_reason
0,1,2024-09-11,monthly,basic,12.99,promo_offer,android,45-54,yes,2025-06-29,technical_issues
1,2,2025-08-14,annual,basic,9.74,promo_offer,web,45-54,no,,
2,3,2025-01-24,monthly,basic,12.99,referral,ios,55+,no,,
3,4,2024-08-15,monthly,basic,12.99,paid_social,ios,25-34,yes,2024-08-20,not_using_enough
4,5,2025-06-24,annual,plus,14.99,paid_social,android,55+,no,,


In [24]:
rows = []
for c in subs_raw.columns:
    v = subs_raw[c]
    rows.append({
        "column": c,
        "blank": (v.str.strip() == "").sum(),
        "padded_whitespace": (v != v.str.strip()).sum(),
        "distinct": v.nunique(),
    })
pd.DataFrame(rows)

,column,blank,padded_whitespace,distinct
0,customer_id,0,0,4200
1,signup_date,0,0,487
2,plan_type,0,0,2
3,plan_tier,0,0,6
4,monthly_price,0,0,6
5,acquisition_channel,0,0,5
6,primary_device,0,0,3
7,age_group,0,0,5
8,churned,0,0,2
9,churn_date,2432,0,464


In [25]:
for c in ["plan_type", "plan_tier", "acquisition_channel", "primary_device", "age_group", "churned", "cancel_reason"]:
    print(f"--- {c}")
    print(subs_raw[c].value_counts().to_string())
    print()

--- plan_type
plan_type
monthly    2661
annual     1539

--- plan_tier
plan_tier
basic      1929
plus       1462
premium     708
Basic        47
Plus         39
Premium      15

--- acquisition_channel
acquisition_channel
organic        1298
paid_social     989
promo_offer     949
referral        488
partner         476

--- primary_device
primary_device
ios        1976
android    1550
web         674

--- age_group
age_group
25-34    1400
35-44    1073
18-24     717
45-54     619
55+       391

--- churned
churned
no     2432
yes    1768

--- cancel_reason
cancel_reason
                     2814
not_using_enough      570
too_expensive         322
found_alternative     175
other                 162
technical_issues      157



In [26]:
subs = subs_raw.copy()

n_case = (subs.plan_tier != subs.plan_tier.str.lower()).sum()
subs["plan_tier"] = subs.plan_tier.str.strip().str.lower()
record("subscriptions", "plan_tier has inconsistent capitalisation", n_case,
       "lower-cased to basic/plus/premium")

assert set(subs.plan_tier) == {"basic", "plus", "premium"}

[subscriptions] plan_tier has inconsistent capitalisation: 101 rows -> lower-cased to basic/plus/premium


In [27]:
print("duplicate customer_id :", subs.customer_id.duplicated().sum())
print("fully duplicated rows: ", subs.duplicated().sum())

attrs = [c for c in subs.columns if c != "customer_id"]
lookalike = subs[subs.duplicated(attrs, keep=False)]
print("rows sharing all attributes except id:", len(lookalike))
print("of which churned = yes               :", (lookalike.churned == "yes").sum())

duplicate customer_id : 0
fully duplicated rows:  0
rows sharing all attributes except id: 56
of which churned = yes               : 0


In [28]:
subs["customer_id"] = subs.customer_id.astype(int)
subs["monthly_price"] = subs.monthly_price.astype(float)
subs["signup_date"] = pd.to_datetime(subs.signup_date)
subs["churn_date"] = pd.to_datetime(subs.churn_date.replace("", np.nan))
subs["cancel_reason"] = subs.cancel_reason.replace("", np.nan)
subs["churned"] = subs.churned.map({"yes": True, "no": False})
assert subs.churned.notna().all()

checks = {
    "churned=yes but no churn_date": ((subs.churned) & subs.churn_date.isna()).sum(),
    "churned=no but has churn_date": ((~subs.churned) & subs.churn_date.notna()).sum(),
    "churned=no but has cancel_reason": ((~subs.churned) & subs.cancel_reason.notna()).sum(),
    "churn_date before signup_date": (subs.churn_date < subs.signup_date).sum(),
    "churn_date after data end": (subs.churn_date > DATA_END).sum(),
    "signup_date after data end": (subs.signup_date > DATA_END).sum(),
    "same-day cancellations": (subs.churn_date == subs.signup_date).sum(),
}
pd.Series(checks, name="count")

churned=yes but no churn_date       0
churned=no but has churn_date       0
churned=no but has cancel_reason    0
churn_date before signup_date       0
churn_date after data end           0
signup_date after data end          0
same-day cancellations              7
Name: count, dtype: int64

In [29]:
price_check = subs.groupby(["plan_type", "plan_tier"]).monthly_price.nunique()
print(price_check.to_string())
assert (price_check == 1).all()

print("\nsignup range:", subs.signup_date.min().date(), "to", subs.signup_date.max().date())
print("churn range :", subs.churn_date.min().date(), "to", subs.churn_date.max().date())

plan_type  plan_tier
annual     basic        1
           plus         1
           premium      1
monthly    basic        1
           plus         1
           premium      1

signup range: 2024-07-01 to 2025-10-30
churn range : 2024-07-02 to 2025-12-31


In [30]:
skipped = subs.churned & subs.cancel_reason.isna()
record("subscriptions", "churners who skipped the cancel-reason question", skipped.sum(),
       "labelled 'not_provided' (active customers stay null)")
subs.loc[skipped, "cancel_reason"] = "not_provided"

[subscriptions] churners who skipped the cancel-reason question: 382 rows -> labelled 'not_provided' (active customers stay null)


In [31]:
usage = usage_raw.copy()

blank_counts = {c: int((usage[c].str.strip() == "").sum()) for c in usage.columns}
print("blank cells per column:", blank_counts)

for c in ["customer_id", "workouts_completed", "minutes_active",
          "classes_booked", "support_tickets"]:
    usage[c] = pd.to_numeric(usage[c].replace("", np.nan))

assert usage.month.str.fullmatch(r"\d{4}-\d{2}").all()
usage.describe().round(2)

blank cells per column: {'customer_id': 0, 'month': 0, 'workouts_completed': 0, 'minutes_active': 506, 'classes_booked': 0, 'support_tickets': 0}


,customer_id,workouts_completed,minutes_active,classes_booked,support_tickets
count,30685.00,30685.00,30179.00,30685.00,30685.00
mean,2108.97,4.46,134.05,0.72,0.13
std,1205.74,4.10,131.17,1.43,0.37
min,1.00,0.00,0.00,0.00,0.00
25%,1060.00,1.00,40.00,0.00,0.00
50%,2122.00,3.00,100.00,0.00,0.00
75%,3143.00,6.00,189.00,1.00,0.00
max,4200.00,48.00,1653.00,16.00,4.00


In [32]:
dup_mask = usage.duplicated(keep="first")
key_dups = usage.duplicated(["customer_id", "month"]).sum()
print("exact duplicate rows         :", dup_mask.sum())
print("repeated (customer, month)   :", key_dups)
print("same count (all are copies)? :", key_dups == dup_mask.sum())

record("usage", "exact duplicate customer-month rows", dup_mask.sum(),
       "kept first copy, dropped the rest")
usage = usage[~dup_mask].reset_index(drop=True)

assert not usage.duplicated(["customer_id", "month"]).any()
print("rows after dedup:", len(usage))

exact duplicate rows         : 160
repeated (customer, month)   : 160
same count (all are copies)? : True
[usage] exact duplicate customer-month rows: 160 rows -> kept first copy, dropped the rest
rows after dedup: 30525


In [33]:
obs = usage.dropna(subset=["minutes_active"])

print("workouts = 0 but minutes > 0:", ((obs.workouts_completed == 0) & (obs.minutes_active > 0)).sum())
print("workouts > 0 but minutes = 0:", ((obs.workouts_completed > 0) & (obs.minutes_active == 0)).sum())

ratio = (obs.minutes_active / obs.workouts_completed)[obs.workouts_completed > 0]
print("\nminutes per workout:")
print(ratio.describe().round(1).to_string())

blank = usage.minutes_active.isna()
print("\nblank rows            :", blank.sum())
print("  with workouts = 0   :", (blank & (usage.workouts_completed == 0)).sum())
print("  with workouts > 0   :", (blank & (usage.workouts_completed > 0)).sum())

workouts = 0 but minutes > 0: 0
workouts > 0 but minutes = 0: 0

minutes per workout:
count    26832.0
mean        30.0
std          6.9
min         18.0
25%         24.0
50%         30.0
75%         36.0
max         42.0

blank rows            : 503
  with workouts = 0   : 55
  with workouts > 0   : 448


In [34]:
blank = usage.minutes_active.isna()
median_ratio = ratio.median()

usage["minutes_imputed"] = blank
usage.loc[blank, "minutes_active"] = np.where(
    usage.loc[blank, "workouts_completed"] == 0,
    0,
    usage.loc[blank, "workouts_completed"] * median_ratio,
)
usage["minutes_active"] = usage.minutes_active.round().astype(int)

record("usage", "blank minutes_active", blank.sum(),
       f"imputed (0 if no workouts, else workouts x {median_ratio:.0f}); flagged in minutes_imputed")
assert usage.minutes_active.notna().all()

[usage] blank minutes_active: 503 rows -> imputed (0 if no workouts, else workouts x 30); flagged in minutes_imputed


In [35]:
tier = subs.set_index("customer_id").plan_tier
usage["plan_tier"] = usage.customer_id.map(tier)

print(usage.groupby("plan_tier").classes_booked.agg(
    rows="size",
    rows_with_classes=lambda s: (s > 0).sum(),
    mean="mean",
).round(2))

assert (usage.loc[usage.plan_tier == "basic", "classes_booked"] == 0).all()
usage["classes_eligible"] = usage.plan_tier != "basic"
usage = usage.drop(columns="plan_tier")

            rows  rows_with_classes  mean
plan_tier                                
basic      14447                  0  0.00
plus       10873               6006  1.14
premium     5205               3512  1.85


In [36]:
print("usage customers not in subscriptions:", (~usage.customer_id.isin(subs.customer_id)).sum())
print("subscribers with no usage rows      :", (~subs.customer_id.isin(usage.customer_id)).sum())

usage["month_idx"] = usage.month.str[:4].astype(int) * 12 + usage.month.str[5:7].astype(int)
month_number = lambda d: d.dt.year * 12 + d.dt.month

subs["signup_idx"] = month_number(subs.signup_date)
subs["end_idx"] = month_number(subs.churn_date.fillna(DATA_END))   # churn month, or Dec 2025 if active

j = usage.merge(subs[["customer_id", "signup_idx", "end_idx"]], on="customer_id")
print("usage rows before signup month:", (j.month_idx < j.signup_idx).sum())
print("usage rows after churn month  :", (j.month_idx > j.end_idx).sum())

usage customers not in subscriptions: 0
subscribers with no usage rows      : 0
usage rows before signup month: 0
usage rows after churn month  : 0


In [37]:
subs["usage_months_expected"] = subs.end_idx - subs.signup_idx + 1

g = usage.groupby("customer_id").agg(
    usage_months_observed=("month_idx", "size"),
    first_usage_idx=("month_idx", "min"),
)
subs = subs.merge(g, on="customer_id", how="left")
subs["months_missing"] = subs.usage_months_expected - subs.usage_months_observed

print("customers with extra months  :", (subs.months_missing < 0).sum())
print("customers with missing months:", (subs.months_missing > 0).sum())
print("  of which churned = yes     :", ((subs.months_missing > 0) & subs.churned).sum())
print("  of which churned = no      :", ((subs.months_missing > 0) & ~subs.churned).sum())
print("rows expected:", subs.usage_months_expected.sum(), "| rows observed:", len(usage))

early = subs[subs.months_missing > 0]
print("\nusage months present for each of them:")
print(early.usage_months_observed.value_counts().to_string())

customers with extra months  : 0
customers with missing months: 139
  of which churned = yes     : 0
  of which churned = no      : 139
rows expected: 31789 | rows observed: 30525

usage months present for each of them:
usage_months_observed
1    139


In [38]:
subs["usage_stopped_early"] = (subs.months_missing > 0) & ~subs.churned

first_month = (usage.sort_values("month_idx")
                    .groupby("customer_id").first()
                    [["workouts_completed", "minutes_active", "support_tickets"]])

x = subs.merge(first_month, left_on="customer_id", right_index=True)
x["group"] = np.select(
    [x.usage_stopped_early, x.churned],
    ["active, usage stopped after month 1", "churned (recorded)"],
    default="active, full usage",
)

print("FIRST-MONTH behaviour by group (average)")
print(x.groupby("group")[["workouts_completed", "minutes_active", "support_tickets"]].mean().round(2).to_string())

print("\nFlagged customers as a share of ACTIVE customers, by plan type")
print(x[~x.churned].groupby("plan_type").usage_stopped_early.agg(customers="sum", share="mean").round(3).to_string())

FIRST-MONTH behaviour by group (average)
                                     workouts_completed  minutes_active  support_tickets
group                                                                                   
active, full usage                                 4.83          143.95             0.08
active, usage stopped after month 1                2.09           64.92             0.19
churned (recorded)                                 3.03           90.66             0.10

Flagged customers as a share of ACTIVE customers, by plan type
           customers  share
plan_type                  
annual             6  0.005
monthly          133  0.102


In [39]:
attrs = ["signup_date", "plan_type", "plan_tier", "monthly_price", "acquisition_channel",
         "primary_device", "age_group", "churned", "churn_date", "cancel_reason"]

pairs = (subs[subs.duplicated(attrs, keep=False)]
         .groupby(attrs, dropna=False)["customer_id"].apply(list))

def usage_signature(cid):
    rows = usage[usage.customer_id == cid].sort_values("month_idx")
    return tuple(rows[["workouts_completed", "minutes_active", "classes_booked", "support_tickets"]]
                 .itertuples(index=False, name=None))

compared = same = 0
for ids in pairs:
    for other in ids[1:]:
        compared += 1
        same += usage_signature(ids[0]) == usage_signature(other)

print(f"pairs compared: {compared} | identical usage histories: {same}")

pairs compared: 28 | identical usage histories: 0


In [40]:
record("subscriptions", "active customers whose usage stops after month 1 (usage_stopped_early)",
       subs.usage_stopped_early.sum(), "kept as recorded; flagged for sensitivity analysis")

subs_clean = subs[[
    "customer_id", "signup_date", "plan_type", "plan_tier", "monthly_price",
    "acquisition_channel", "primary_device", "age_group", "churned", "churn_date",
    "cancel_reason", "usage_months_expected", "usage_months_observed", "usage_stopped_early",
]].sort_values("customer_id").reset_index(drop=True)

usage_clean = usage[[
    "customer_id", "month", "workouts_completed", "minutes_active", "classes_booked",
    "support_tickets", "minutes_imputed", "classes_eligible",
]].sort_values(["customer_id", "month"]).reset_index(drop=True)

assert subs_clean.customer_id.is_unique and len(subs_clean) == 4200
assert not usage_clean.duplicated(["customer_id", "month"]).any()
assert usage_clean.isna().sum().sum() == 0
assert set(usage_clean.customer_id) == set(subs_clean.customer_id)

subs_clean.to_csv(PROC / "subscriptions_clean.csv", index=False)
usage_clean.to_csv(PROC / "usage_clean.csv", index=False)
print("subscriptions_clean:", subs_clean.shape, "| usage_clean:", usage_clean.shape)

[subscriptions] active customers whose usage stops after month 1 (usage_stopped_early): 139 rows -> kept as recorded; flagged for sensitivity analysis
subscriptions_clean: (4200, 14) | usage_clean: (30525, 8)


In [41]:
log_df = pd.DataFrame(log, columns=["Area", "Issue", "Rows affected", "Action"])

lines = ["| Area | Issue | Rows affected | Action |", "|---|---|---:|---|"]
for area, issue, n, action in log:
    lines.append(f"| {area} | {issue} | {n:,} | {action} |")

passed = [
    "No duplicate customer_id; row counts match the project brief (4,200 / 30,685 raw rows).",
    "churned, churn_date and cancel_reason are consistent; no churn before signup or after 2025-12-31.",
    "monthly_price is fully determined by plan type and tier.",
    "All usage customers exist in subscriptions; no usage before signup or after cancellation.",
    "classes_booked is 0 on every Basic-plan row (structural, not disengagement).",
    "28 look-alike customer pairs have different usage histories, so they are distinct people.",
]

text = ("# Cleaning log\n\n" + "\n".join(lines)
        + "\n\n## Checks that passed\n" + "\n".join(f"- {p}" for p in passed) + "\n")
(REPORTS / "cleaning_log.md").write_text(text)

log_df

,Area,Issue,Rows affected,Action
0,subscriptions,plan_tier has inconsistent capitalisation,101,lower-cased to basic/plus/premium
1,subscriptions,churners who skipped the cancel-reason question,382,labelled 'not_provided' (active customers stay...
2,usage,exact duplicate customer-month rows,160,"kept first copy, dropped the rest"
3,usage,blank minutes_active,503,"imputed (0 if no workouts, else workouts x 30)..."
4,subscriptions,active customers whose usage stops after month...,139,kept as recorded; flagged for sensitivity anal...
